# Concurrent Execution with Fleche

This notebook shows how to use `@fleche`-decorated functions with Python's three executor types.

`BoundWrapper.bind(func)` is the recommended pattern for sharing cache state across workers: it captures the active cache and metadata at bind time and restores them on every subsequent call — even in subprocesses — without requiring a hand-written wrapper function.

| Executor | Required storage | Recommended pattern |
|---|---|---|
| `ThreadPoolExecutor` | Any (Memory or file) | `BoundWrapper.bind(func)` |
| `ProcessPoolExecutor` | **Persistent** (file / SQL) | `BoundWrapper.bind(func)` |
| `SingleNodeExecutor` (executorlib) | **Persistent** (file / SQL) | `BoundWrapper.bind(func)` |

In [ ]:
import time
import contextvars
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from fleche import fleche, cache, BoundWrapper
from fleche.caches import Cache
from fleche.storage.memory import Memory

## 1. ThreadPoolExecutor

### The problem: threads don't inherit the cache context by default

Python threads each start with a *copy* of the context that existed when `ThreadPoolExecutor` was created, not a live reference to the calling thread's context. This means that if you set a cache inside the main thread and submit work to a pool, the workers won't see it.

In [ ]:
@fleche
def add(x, y):
    return x + y

In [ ]:
mem = Memory({})
my_cache = Cache(mem, mem)

with cache(my_cache):
    with ThreadPoolExecutor(max_workers=2) as pool:
        pool.submit(add, 1, 2).result()

# The result was NOT stored in my_cache — the thread used the default cache
print("In my_cache:", my_cache.contains(add.digest(1, 2)))  # False

### The fix: propagate context explicitly with `ctx.run`

Capture the current context with `contextvars.copy_context()` and wrap the call in `ctx.run(...)`. This hands the *exact same context object* to the thread, so the `cache(...)` context manager is visible inside the worker.

In [ ]:
mem = Memory({})
my_cache = Cache(mem, mem)

with cache(my_cache):
    ctx = contextvars.copy_context()          # snapshot the current context
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [
            pool.submit(ctx.run, add, x, x + 1)  # ctx.run propagates the context
            for x in range(4)
        ]
        results = [f.result() for f in futures]

print("Results:", results)                       # [1, 3, 5, 7]
print("In my_cache:", my_cache.contains(add.digest(0, 1)))  # True

### Alternative: `BoundWrapper.bind`

`BoundWrapper.bind(func)` captures the active cache at bind time. The resulting callable
restores that cache on every call — no `ctx.run` is required at the call site. This is
particularly convenient when you want to pass a pre-configured callable around without
threading a context object alongside it.

In [ ]:
mem = Memory({})
my_cache = Cache(mem, mem)

with cache(my_cache):
    bound_add = BoundWrapper.bind(add)   # capture cache context at bind time

# bound_add always uses my_cache — no ctx.run needed at the call site
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(bound_add, x, x + 1) for x in range(4)]
    results = [f.result() for f in futures]

print("Results:", results)                          # [1, 3, 5, 7]
print("In my_cache:", my_cache.contains(add.digest(0, 1)))  # True

### Shared cache across all threads

Because all workers run inside the same context, they share the same in-memory store. Results computed by one thread are immediately visible to every other thread in the pool.

In [ ]:
@fleche
def slow_square(x):
    time.sleep(0.1)   # simulate work
    return x * x

mem = Memory({})
my_cache = Cache(mem, mem)

# First batch — computes and caches
with cache(my_cache):
    ctx = contextvars.copy_context()
    with ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(lambda x: ctx.run(slow_square, x), range(5)))

# Second batch — everything hits the cache (no sleep)
start = time.time()
with cache(my_cache):
    ctx = contextvars.copy_context()
    with ThreadPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(lambda x: ctx.run(slow_square, x), range(5)))

elapsed = time.time() - start
print(f"Results: {results}")
print(f"Second batch took {elapsed:.3f}s (cache hits — no sleep)")

---
## 2. ProcessPoolExecutor

### Why `ctx.run` doesn't work across processes

With `ProcessPoolExecutor`, every argument passed to a worker must be serialised with `pickle` and sent to a new Python interpreter. `contextvars.Context` objects **cannot be pickled**, so the threadpool trick is not available:

```python
# ❌ This raises: TypeError: cannot pickle 'Context' object
ctx = contextvars.copy_context()
executor.submit(ctx.run, add, 1, 2)
```

The good news: `@fleche`-decorated functions *are* picklable (they are module-level names), so you can pass them as worker targets without any issues.

### The pattern: set up cache inside the worker

Each worker process starts with a fresh Python interpreter that has no cache configured. The simplest fix is to wrap your cached call inside a thin helper that creates the cache context *after* the process has spawned.

In [ ]:
# Worker functions must be defined at module level to be picklable

@fleche
def expensive(x):
    return x ** 3

def _worker(x):
    """Thin wrapper that sets up a cache before calling the real function."""
    with cache(Cache(Memory({}), Memory({}))):
        return expensive(x)


with ProcessPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(_worker, range(5)))

print("Results:", results)   # [0, 1, 8, 27, 64]

### Sharing a persistent cache across processes

With in-memory storage each worker gets its own isolated cache. To share results between processes you need a **picklable, on-disk backend** (e.g. file-based storage). Pass the cache object as an argument and re-register it inside the worker:

In [ ]:
import tempfile, os
from fleche.storage.file import PickleFileStorage

tmpdir = tempfile.mkdtemp()
shared_cache = Cache(
    PickleFileStorage(os.path.join(tmpdir, "meta")),
    PickleFileStorage(os.path.join(tmpdir, "results")),
)

def _worker_shared(x, worker_cache):
    with cache(worker_cache):     # re-register the pickled cache object
        return expensive(x)


# First run — computes and persists to disk
with ProcessPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(_worker_shared, x, shared_cache) for x in range(5)]
    results = [f.result() for f in futures]

print("Results:", results)

# Verify results are in the shared cache
print("Cached:", shared_cache.contains(expensive.digest(3)))   # True

### Cleaner pattern: `BoundWrapper.bind`

Instead of writing a separate worker wrapper function and passing directory paths as arguments,
use `BoundWrapper.bind(func)`. It embeds the cache configuration into the callable — the worker
receives the bound callable and automatically uses the correct cache.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    shared_cache = Cache(
        PickleFileStorage(os.path.join(tmpdir, "meta")),
        PickleFileStorage(os.path.join(tmpdir, "results")),
    )

    with cache(shared_cache):
        bound_expensive = BoundWrapper.bind(expensive)   # carry the cache config

    # No separate wrapper function needed — submit bound_expensive directly
    with ProcessPoolExecutor(max_workers=2) as pool:
        results = list(pool.map(bound_expensive, range(5)))

print("Results:", results)                                      # [0, 1, 8, 27, 64]
print("Cached:", shared_cache.contains(expensive.digest(3)))   # True

---
## 3. executorlib.SingleNodeExecutor

[executorlib](https://github.com/pyiron/executorlib) is a third-party library for submitting tasks to HPC schedulers and local process pools. Its `SingleNodeExecutor` uses *process-based* workers with the same isolation semantics as `ProcessPoolExecutor`.

> **Note:** `executorlib` is an optional dependency. Install it with `pip install executorlib`.

`@fleche`-decorated functions are picklable (they are module-level names), so they can be submitted to `SingleNodeExecutor` directly:

In [ ]:
from executorlib import SingleNodeExecutor

@fleche
def cube_sne(x):
    return x ** 3

with SingleNodeExecutor() as executor:
    future = executor.submit(cube_sne, 4)
    result = future.result()

print("Result:", result)  # 64

### In-memory cache is not propagated

Like `ProcessPoolExecutor`, `SingleNodeExecutor` spawns worker processes with fresh Python interpreters. An in-memory cache set in the parent is **not visible** inside the worker — results computed there are stored in the worker's own ephemeral cache and are lost when the process exits.

In [ ]:
mem = Memory({})
my_cache = Cache(mem, mem)

with cache(my_cache):
    with SingleNodeExecutor() as executor:
        future = executor.submit(cube_sne, 5)
        result = future.result()

print("Result:", result)                                # 125
print("In my_cache:", my_cache.contains(cube_sne.digest(5)))  # False — worker results don't reach the parent

### Sharing results with file-backed storage

To persist results across processes, use a **file-backed storage backend** and point the worker at the same on-disk directory. The worker sets up a fresh `Cache` around the file-backed store; results written to disk are immediately visible to any process that reads from the same path.

In [ ]:
def _sne_worker_shared(x, values_dir, calls_dir):
    """Worker that wires up a shared file-backed cache then calls the fleche function."""
    worker_cache = Cache(
        PickleFileStorage(os.path.join(values_dir)),
        PickleFileStorage(os.path.join(calls_dir)),
    )
    with cache(worker_cache):
        return cube_sne(x)


with tempfile.TemporaryDirectory() as tmpdir:
    values_dir = os.path.join(tmpdir, "values")
    calls_dir  = os.path.join(tmpdir, "calls")

    parent_cache = Cache(
        PickleFileStorage(values_dir),
        PickleFileStorage(calls_dir),
    )

    with SingleNodeExecutor() as executor:
        futures = [executor.submit(_sne_worker_shared, x, values_dir, calls_dir) for x in range(5)]
        results = [f.result() for f in futures]

print("Results:", results)                                      # [0, 1, 8, 27, 64]
print("In parent_cache:", parent_cache.contains(cube_sne.digest(3)))  # True

### Cleaner pattern: `BoundWrapper.bind`

Instead of writing a separate worker function that recreates the cache from directory arguments,
use `BoundWrapper.bind(func)`. It embeds the cache configuration into the callable itself —
no extra arguments need to be threaded through to the worker.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    values_dir = os.path.join(tmpdir, "values")
    calls_dir  = os.path.join(tmpdir, "calls")

    parent_cache = Cache(
        PickleFileStorage(values_dir),
        PickleFileStorage(calls_dir),
    )

    with cache(parent_cache):
        bound_cube = BoundWrapper.bind(cube_sne)   # carry the cache config

    # No separate wrapper function needed — submit bound_cube directly
    with SingleNodeExecutor() as executor:
        futures = [executor.submit(bound_cube, x) for x in range(5)]
        results = [f.result() for f in futures]

print("Results:", results)                                          # [0, 1, 8, 27, 64]
print("In parent_cache:", parent_cache.contains(cube_sne.digest(3)))  # True

### Summary

```
ThreadPoolExecutor
  Default (no propagation)          →  workers use the default cache, NOT your custom one ❌
  With ctx.run                       →  workers share the same cache as the main thread ✅
  With BoundWrapper.bind(func)       →  same effect, no ctx.run needed at call site ✅

ProcessPoolExecutor
  In-memory cache (parent-side)      →  isolated per-worker caching, parent cannot read results ❌
  File-backed + BoundWrapper.bind()  →  shared persistent cache, no wrapper function needed ✅
  ctx.run                            →  not possible (Context not picklable) ❌

executorlib.SingleNodeExecutor
  In-memory cache (parent-side)      →  not visible in worker processes ❌
  File-backed + BoundWrapper.bind()  →  worker results visible to parent ✅
  ctx.run                            →  not possible (process-based) ❌
```

**Storage reference**

| Backend | Cross-process sharing |
|---|---|
| `Memory` | ❌ Ephemeral — process-local only |
| `PickleFile` | ✅ Persistent — shared via filesystem |
| `Sql` | ✅ Persistent — shared via database |
| `BagOfHolding` | ✅ Persistent — shared via HDF5 file |